## Specialized control

A stored one-bit specialization lets the compiler control the central X of
`T† X T` while T and T† still act directly on the target. A component can
provide separate positive and negative specializations. Multiple controls
can compute a shared predicate, use the positive body, and erase the predicate
with measurement and feedback. Controls and the target pass through.

In [ ]:
from collections import Counter

from IPython.display import SVG, display

from examples.algorithms.specialized_control_catalog import (
    SpecializedControlExample,
    controlled_adjoint_phase_conjugated_x,
    doubly_controlled_phase_conjugated_x,
    phase_conjugated_x_program,
)
from flagquantum.compiler._component_trace.program_call_graph import program_component_call_graph
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.component_score import component_register_score
from flagquantum.drawer.register_graph import export_register_graph_dot
from flagquantum.dynamic import DynamicCircuit

positive = phase_conjugated_x_program(positive_control=True)
double = doubly_controlled_phase_conjugated_x()

In [ ]:
score = component_register_score(double)
display(SVG(render_call_graph_svg(export_register_graph_dot(score))))
graph = program_component_call_graph(double, max_depth=2)
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
double_gates = DynamicCircuit(4).append(SpecializedControlExample(double, 2)).to_ir().instructions
display(Counter(gate.name for gate in double_gates))

## Controlling an adjoint

Generic control selects the inverse first and controls its individual
operations, so it also controls both phase gates. The source tutorial
constructs this generic wrapper. Its uncontrolled X body uses incompatible
port names, so decomposing that wrapper in Qualtran v0.7.0 raises a binding
error. The native generic example executes the intended gate sequence.

In [ ]:
generic = controlled_adjoint_phase_conjugated_x(specialize=False)
gates = DynamicCircuit(2).append(SpecializedControlExample(generic, 1)).to_ir().instructions
display([gate.name for gate in gates])

A specialized adjoint selects the positive controlled body and then reverses
it. The phases remain unconditional. The selected path has the same
operation as the generic one and uses two T-family gates and one CNOT.

In [ ]:
selected = controlled_adjoint_phase_conjugated_x(specialize=True)
display(program_component_call_graph(selected).leaf_counts)

In [ ]:
selected_gates = DynamicCircuit(2).append(SpecializedControlExample(selected, 1)).to_ir().instructions
display([gate.name for gate in selected_gates])

In [ ]:
positive_gates = DynamicCircuit(2).append(SpecializedControlExample(positive, 1)).to_ir().instructions
assert selected_gates == positive_gates  # This phase-conjugated X is self-adjoint.